###Data Reading JSON

In [0]:
df_json = spark.read.format('json')\
    .option('inferSchema',True)\
    .option('header',True)\
    .option('multiLine',False)\
    .load('/Volumes/sparttable/default/saprk_shru/drivers.json')    
    

In [0]:
df_json.display()


### Data Reading

In [0]:
dbutils.fs.ls('/Volumes/sparttable/default//saprk_shru')

In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load('/Volumes/sparttable/default/saprk_shru/BigMart Sales.csv')

In [0]:
df.display()

##Schema Definition

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
         Item_Identifier string,
         Item_Weight STRING,
         Item_Fat_Content string,
         Item_Visibility double,
         Item_Type string,
         Item_MRP double,
         Outlet_Identifier string,
         Outlet_Establishment_Year integer,
         Outlet_Size string,
         Outlet_Location_Type string,
         Outlet_Type string,
         Item_Outlet_Sales double
'''

In [0]:
df = spark.read.format('csv')\
    .schema(my_ddl_schema)\
    .option('header',True)\
    .load('/Volumes/sparttable/default/saprk_shru/BigMart Sales.csv')


In [0]:
df.display()

In [0]:
df.printSchema()

### StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [0]:
my_strct_schema = StructType([
                                StructField('Item_Identifier',StringType(),True),
                                StructField('Item_Weight',StringType(),True),
                                StructField('Item_Fat_Content',StringType(),True),
                                StructField('Item_Visibility',StringType(),True),
                                StructField('Item_Type',StringType(),True),
                                StructField('Item_MRP',StringType(),True),
                                StructField('Outlet_Identifier',StringType(),True),
                                StructField('Outlet_Establishment_Year',StringType(),True),
                                StructField('Outlet_Size',StringType(),True),
                                StructField('Outlet_Location_Type',StringType(),True),
                                StructField('Outlet_Type',StringType(),True),
                                StructField('Item_Outlet_Sales',StringType(),True   )

])

In [0]:
df = spark.read.format('csv')\
    .schema(my_strct_schema)\
    .option('header',True)\
    .load('/Volumes/sparttable/default/saprk_shru/BigMart Sales.csv')
df.printSchema()
df.display()


In [0]:
df.printSchema()

###SELECT

In [0]:
df.display()

In [0]:
df_sel = df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content'))
df_sel.display()


###ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()


###FILTER

####Scenario - 1

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

####Scenario - 2

In [0]:
df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight').cast('double')<10)).display()

####Scenario - 3

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin(['Tier 1','Tier 2']))).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_wt').display()

### withColumn

####Scenario - 1

In [0]:
df = df.withColumn('flag',lit("new")).display()

In [0]:
df.withColumn('multiply',col('Item_Weight').cast('double')*col('Item_MRP').cast('double')).display()

#### Scenario - 2

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","L F")).display()

### Type Casting

In [0]:
df = df.withColumn('Item_Weight',col("Item_Weight").cast(StringType()))

### Sort

####Scenario - 1 

In [0]:
df.sort(col('Item_Weight').desc()).display()

#### Scenario -2

In [0]:
df.sort(col('Item_Visibility').asc()).display()


#### Sceanrio - 3

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending = [0,0]).display()

#### Scenario - 4

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending = [0,1]).display()

### LIMIT

In [0]:
df.limit(10).display()

### DROP

#### Scenario - 1

In [0]:
df.drop('Item_Visibility').display()

#### Scenario - 2

In [0]:
df.drop('Item_Visibility','Item_Type').display()

### DROP_DUPLICATES

In [0]:
df.dropDuplicates().display()

#### Scenario - 2

In [0]:
df.dropDuplicates(subset = ['Item_Type']).display()

In [0]:
df.distinct().display()

###UNION and UNION BYNAME

Preparing Dataframes

In [0]:
data1 = [('1,','kad'),
         ('2','sid')]
schema1 = 'id STRING,name STRING'

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
         ('4','jas')]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2,schema2)
          

####UNION

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad','1'),
         ('sid','2')]
schema1 = 'name STRING,id STRING'

df1 = spark.createDataFrame(data1,schema1)

####UNION BY NAME

In [0]:
df1.unionByName(df2).display()

###STRING FUNCTIONS

####Initcap()

In [0]:
df.select(upper('Item_Type').alias('upper_Item_Type')).display()

###DATE FUNCTIONS

#### Current Date

In [0]:
df = df.withColumn('curr_date',current_date())

df.display()

#### Date_Add()

In [0]:
df = df.withColumn('week_after',date_add('curr_date',7))

df.display()

####Date_Sub()

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()

In [0]:
df = df.withColumn('week Before',date_add('curr_date',-7))
df.display()

### Date Diff

In [0]:
df = df.withColumn('datediff',datediff('week_after','curr_date'))
df.display()

###DATE_FORMAT

In [0]:
df = df.withColumn('week Before',date_format('week Before','dd-MM-yyyy'))

df.display()

###HANDLING NULLS

####Droping Nulls

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

#### Filling Nulls

In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable',subset=['Outlet_Size']).display()

###SPLIT FUNCTION

####SPLIT

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

####Indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

###EXPLODE

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### ARRAY_CONTAINS

In [0]:
df_exp.display()

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

###Group By

####Scenario - 1

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

####Scenario - 2

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP').alias('AVG_ITEM_MRP')).display()

####Scenario - 3 

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum('Item_MRP').alias('Total_MRP')).display()

####Scenario - 4

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum("Item_MRP").alias("Total_MRP"),avg('Item_MRP').alias('AVG_MRP')).display()

###COLLECT_LIST

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string , book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book').alias('books')).display()

###PIVOT

In [0]:
df.groupBy('Item_Type').pivot("Outlet_Size").agg(avg("Item_MRP")).display()

### WHEN PTHERWISE

#### Scenario - 2

In [0]:
df = df.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non_veg').otherwise('Veg'))

df.display()


### Scenario - 2

In [0]:
df.withColumn('veg_exp_flag',when((col('veg_flag')=='Veg') & (col('Item_MRP').cast('double')<100),'Veg_InExpensive')\
    .when((col('veg_flag')=='Veg') & (col('Item_MRP').cast('double')>100),'Veg_Expensive')\
    .otherwise('Non_Veg')).display()


### JOINS

In [0]:
dataj1 = [('1', 'gaur', 'd01'),
          ('2', 'kit', 'd02'),
          ('3', 'paul', 'd03'),
          ('4', 'john', 'd03'),
          ('5', 'paul', 'd05'),
          ('6', 'paul', 'd06')  ]

schemaj1 ='empid string, name string, deptid string'

df1 = spark.createDataFrame(dataj1,schemaj1)

df1.display()
dataj2 = [('d01', 'HR'),
          ('d02', 'Marketing'),
          ('d03', 'Accounts'),
          ('d04', 'IT'),
          ('d05', 'Finance')]

schemaj2 = 'deptid string, deptname string'

df2 = spark.createDataFrame(dataj2,schemaj2)

df2.display()


####Inner Join

In [0]:
df1.join(df2,df2.deptid==df1.deptid,'inner').display()

#### Left Join

In [0]:
df1.join(df2,df1.deptid==df2.deptid,'left').display()

### Right JOIN

In [0]:
df1.join(df2,df1.deptid==df2.deptid,'right').display()


#### ANTI JOIN

In [0]:
df1.join(df2,df1.deptid==df2.deptid,'anti').display()


#### Full JOIN

In [0]:
df1.join(df2,df1.deptid==df2.deptid,'full').display()

In [0]:
df1.join(df2,df1.deptid==df2.deptid,'cross').display()
df1.crossJoin(df2).display()

### WINDOW FUNCTION

#### ROW NUMBER

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

#### RANK()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Outlet_Sales')).partitionBy(col('Item_Identifier'))))\
             .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_identifier')))).display()

#### Cumulative SUM


In [0]:
df.withColumn('cum sum', sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cum sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

### USE DEFINED FUNCTIONS(UDF)

#### Step - 1

In [0]:
def my_func(x):
    return x*x

####STep - 2

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol',my_udf(col('Item_MRP'))).display()

### DATA WRITING

#### CSV(comma sepearted value)

In [0]:
df.write.format('csv')\
    .save('/Volumes/sparttable/default/saprk_shru/mycsv/data.csv')

#### APPEND

In [0]:
df.write.format('csv')\
    .mode('append')\
    .save('/Volumes/sparttable/default/saprk_shru/mycsv/data.csv')

####OVERWRITE

In [0]:
df.write.format('csv')\
    .mode('overwrite')\
    .option('path','/Volumes/sparttable/default/saprk_shru/mycsv/data.csv')\
    .save()

####ERROR

In [0]:
df.write.format('csv')\
    .mode('error')\
    .option('path','/Volumes/sparttable/default/saprk_shru/mycsv/data.csv')\
    .save()

####IGONRE

In [0]:
df.write.format('csv')\
    .mode('ignore')\
    .option('path','/Volumes/sparttable/default/saprk_shru/mycsv/data.csv')\
    .save()

#### PARQUET

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path','/Volumes/sparttable/default/saprk_shru/mycsv/data.csv')\
    .save()

####Table


In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable('my_table')

In [0]:
df.display()

### SPARK SQL

#### CreateTempView 

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view where Item_Fat_Content = 'Low Fat'

In [0]:
df_sql = spark.sql('select * from my_view')
df_sql.display()